# Setup

In [ ]:
!pip install wandb
!apt-get install poppler-utils
!pip install pdf2image
import requests
from pdf2image import convert_from_path
import matplotlib.pyplot as plt
import numpy as np
import torch
import requests
from torchvision import *
import torchvision.transforms as transforms
from torchvision.models import *
import wandb as wb

In [ ]:
def GPU(data):
    return torch.tensor(data, requires_grad=True, dtype=torch.float, device=torch.device('cuda'))

def GPU_data(data):
    return torch.tensor(data, requires_grad=False, dtype=torch.float, device=torch.device('cuda'))

def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x, cmap = 'gray')
    ax.axis('off')
    fig.set_size_inches(5, 5)
    plt.show()

def get_google_slide(url):
    url_head = "https://docs.google.com/presentation/d/"
    url_body = url.split('/')[5]
    page_id = url.split('.')[-1]
    return url_head + url_body + "/export/pdf?id=" + url_body + "&pageid=" + page_id

def get_slides(url):
    url = get_google_slide(url)
    r = requests.get(url, allow_redirects=True)
    open('file.pdf', 'wb').write(r.content)
    images = convert_from_path('file.pdf', 500)
    return images

def load(image, size=224):
    means = [0.485, 0.456, 0.406]
    stds = [0.229, 0.224, 0.225]
    transform = transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(means, stds)
    ])
    tensor = transform(image).unsqueeze(0).to(device)
    tensor.requires_grad = True
    return tensor

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load Alexnet

In [ ]:
labels = {int(key):value for (key, value) in requests.get('https://s3.amazonaws.com/mlpipes/pytorch-quick-start/labels.json').json().items()}
model = alexnet(pretrained=True).to(device)
model.eval();

# Load Images

In [ ]:
url = "https://docs.google.com/presentation/d/1WVE287B4LBI3dZOvofhCrzpc8pP8PWHan8WUhLB9swA/edit#slide=id.g206f8279a60_0_0"

In [ ]:
images = []

for image in get_slides(url):

    plot(image)

    images.append(load(image))

images = torch.vstack(images)

In [ ]:
labels

In [ ]:
images.shape

# Process Image

In [ ]:
y = model(images)

In [ ]:
y.shape

In [ ]:
y_out = y.cpu().detach().numpy()

In [ ]:
plt.plot(y_out[0,:])

In [ ]:
guesses = torch.argmax(y, 1).cpu().numpy()

In [ ]:
guesses.shape

In [ ]:
guesses

In [ ]:
for j,i in enumerate(list(guesses)):
    print(j,i,labels[i])

# Retrain

In [ ]:
Y = np.zeros(50,)
Y[25:] = 1

In [ ]:
Y

In [ ]:
X = y.detach().cpu().numpy()

In [ ]:
X.shape

In [ ]:
plt.plot(X[0],'.')

In [ ]:
X[0]

In [ ]:
np.argmax(X[0])

In [ ]:
index_winner = np.argmax(X[0])

labels[index_winner]

In [ ]:
top_ten = np.argsort(X[0])[::-1][0:10]

In [ ]:
for i in top_ten:
    print(labels[i])

In [ ]:
labels

In [ ]:
plt.hist(X[0])

In [ ]:
X = GPU_data(X)
Y = GPU_data(Y)

In [ ]:
X.shape

In [ ]:
!nvidia-smi

In [ ]:
def softmax(x):
    s1 = torch.exp(x - torch.max(x,1)[0][:,None])
    s = s1 / s1.sum(1)[:,None]
    return s

In [ ]:
def cross_entropy(outputs, labels):
    return -torch.sum(softmax(outputs).log()[range(outputs.size()[0]), labels.long()])/outputs.size()[0]

In [ ]:
def randn_trunc(s): #Truncated Normal Random Numbers
    mu = 0
    sigma = 0.1
    R = stats.truncnorm((-2*sigma - mu) / sigma, (2*sigma - mu) / sigma, loc=mu, scale=sigma)
    return R.rvs(s)

In [ ]:
def Truncated_Normal(size):

    u1 = torch.rand(size)*(1-np.exp(-2)) + np.exp(-2)
    u2 = torch.rand(size)
    z  = torch.sqrt(-2*torch.log(u1)) * torch.cos(2*np.pi*u2)

    return z

In [ ]:
def acc(out,y):
    with torch.no_grad():
        return (torch.sum(torch.max(out,1)[1] == y).item())/y.shape[0]

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
a = np.array([[1, 2], [3, 4]])

In [ ]:
b = np.array([[5, 6], [1, 0]])

In [ ]:
a.shape,b.shape

In [ ]:
a@b

In [ ]:
def inner_prod(a,b):
    c = 0
    for i in range(a.shape[0]):
        c += a[i]*b[i]
    return c

In [ ]:
def mat_mult(a,b):

    if a.shape[1] == b.shape[0]:

        c = np.zeros((a.shape[0],b.shape[1]))

        for i in range(a.shape[0]):
            for j in range(b.shape[1]):
               c[i,j] = inner_prod(a[i,:],b[:,j])
        return c

    else:
        print("error")



In [ ]:
%%timeit
a@b

In [ ]:
%%timeit
mat_mult(a,b)

In [ ]:
x = np.random.random((1000,1))

In [ ]:
w = np.random.random((2,1000))

In [ ]:
w.shape,x.shape

In [ ]:
z = mat_mult(w,x)

In [ ]:
z.shape

In [ ]:
z

In [ ]:
def get_batch(mode):
    b = c.b
    if mode == "train":
        r = np.random.randint(X.shape[0]-b)
        x = X[r:r+b,:]
        y = Y[r:r+b]
    elif mode == "test":
        r = np.random.randint(X_test.shape[0]-b)
        x = X_test[r:r+b,:]
        y = Y_test[r:r+b]
    return x,y

In [ ]:
def model(x,w):

    return x@w[0]

In [ ]:
def make_plots():

    acc_train = acc(model(x,w),y)

    # xt,yt = get_batch('test')

    # acc_test = acc(model(xt,w),yt)

    wb.log({"acc_train": acc_train})

In [ ]:
wb.init(project="Linear_Model_Photo_1");
c = wb.config

c.h = 0.001
c.b = 32
c.epochs = 100000

w = [GPU(Truncated_Normal((1000,2)))]

optimizer = torch.optim.Adam(w, lr=c.h)

for i in range(c.epochs):

    x,y = get_batch('train')

    loss = cross_entropy(softmax(model(x,w)),y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    wb.log({"loss": loss})

    make_plots()


https://tensorspace.org/